In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-01 12:37:07--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-04-01 12:37:07 (35.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open('input.txt', mode='r', encoding='utf-8') as f:
  text = f.read()

EXERCISES:

EX1: The n-dimensional tensor mastery challenge: Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel, treating the heads as another batch dimension (answer is in nanoGPT).





In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

chars = list(sorted(set(text)))
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
vocab_size = len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")


class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    # How to process all heads in parallel
    # [B, T, C] [C, H]
    # [B, T, NH, C] [C, H] -> [B, T, NH, H] and then reshape [B, T, NH*H]

    self.NH = num_heads
    self.Q = torch.randn(self.NH, n_embd, head_size)
    self.K = torch.randn(self.NH, n_embd, head_size)
    self.V = torch.randn(self.NH, n_embd, head_size)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    self.projection = nn.Linear(n_embd, n_embd)


  def forward(self, x):
    B,T,C = x.shape
    NH = self.NH
    #print(f"Initial input shape {x.shape}")
    x = x.unsqueeze(1).expand(B, NH, T, C)
    #print(f"After input reshape {x.shape}")
    query = x @ self.Q # [B, NH, T, C] @ [B, NH, C, H] -> [B, NH, T, H]
    key = x @ self.K   # [B, NH, T, H]
    value = x @ self.V # [B, NH, T, H]
    #print(f"QKV Shapes {query.shape} {key.shape} {value.shape}")


    key_T = key.transpose(-2, -1)
    #print(f"K transpose {key_T.shape}")

    wei = query @  key_T # [B, NH, T, H] @ [B, NH, H, T] -> [B, NH, T, T]
    wei = wei * (C**-0.5) # Scaling down to approximately be unit gaussian
    # Until this wei is initial affinities based on data


    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1) # Last embedding dimension
    out = wei @ value # [B, NH, T, T] @ [B, NH, T, H] -> [B, NH, T, H]
    #print(f"Wei {wei.shape} out {out.shape}")
    out = out.permute(0,2,1,3) # Change to [B, T, NH, C]
    #print(f"After permute {out.shape}")
    out = out.reshape(B, T, -1) # [B, T, n_embd]
    # Alternate
    """
    out = out.transpose(1, 2)
    out = out.contiguous().view(B, T, C)
    """
    #print(f"After reshape {out.shape}")
    out = self.projection(out) # [B, T, n_embd]
    #print(f"==============================")
    return out


class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)


class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln(x)) # [B, T, C]
    x = x + self.ffwd(self.ln(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      out.append(next_idx.item())
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)


DEVICE is cpu
Train loss 4.296173095703125 Eval loss 4.295577526092529
Train loss 2.9746837615966797 Eval loss 2.9941678047180176
Train loss 2.648902177810669 Eval loss 2.645638942718506
Train loss 2.5130882263183594 Eval loss 2.528024673461914
Train loss 2.4540212154388428 Eval loss 2.4675488471984863
Train loss 2.4055440425872803 Eval loss 2.428790807723999
Train loss 2.343580722808838 Eval loss 2.3773133754730225
Train loss 2.331392288208008 Eval loss 2.3461482524871826
Train loss 2.305638313293457 Eval loss 2.313383102416992
Train loss 2.271836757659912 Eval loss 2.292079448699951
Train loss 2.24548602104187 Eval loss 2.262554883956909
Train loss 2.215745449066162 Eval loss 2.254636526107788
Train loss 2.196986675262451 Eval loss 2.2275583744049072
Train loss 2.1812398433685303 Eval loss 2.217576503753662
Train loss 2.1736643314361572 Eval loss 2.1961750984191895
Train loss 2.1604344844818115 Eval loss 2.203122615814209
Train loss 2.1276040077209473 Eval loss 2.1824398040771484
Tra

In [ ]:
# Generation : Data generation post some training
start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(start_phrase, num_tokens=1_000)

=========== Generated output ==========

PLAUCENAPIUS:
Commed thee?

ANGHERS:
As tisenase
Of the tuch shomere an not: peew dent mracions should ogn do diat.
Dords.

QUEEN ELLIZ:
He that forew Palinctles!

CAMILO:
Behenl.

Fistly kioold his con:
Or sme on rake it, thetser, But.

RORIA:
Calit erise lionsiuss oyou trime what peake atcle breat herow that theird pain, andose! that mercund listed now to hather's staNGed leather, greatice,
With, to eare.

GLRA:
I wituless joest, his from dupood againd
lows, much onow make eprdy. Caladionginurse is shall it to lied
ateds twence!

Sereatern ablesty; and a voutold grow
Which soaimmbenrave ellow:
Norr mearicely of yeedrd necun betcio's bety shall,
Or Jbroviven busy goar with good.
Hen so blep provess.
WI for's there bet meriln me not, bet wortery musty Juine cutt
Thone a grif-starcantine of tresst, Nook ninst:
Where me?
Seco,
Herels mort they
So Cyould, Dentarge then warriciubund what of thook pay none calais, eace he ampeak.

DUKE OF A:
Shank Go

In [ ]:
import torch
## Cat individual heads
x1 = torch.randn(3, 4, 8, 16)
print(x1[0,:,0,:])
x2 = x1.permute(0,2,1,3)
x3 = x2.reshape(3, 8, -1)
print(x3.shape)
print(x3[0, 0])

tensor([[-0.0898,  1.8181, -1.5942, -2.0354, -1.5799,  0.5648,  0.3598, -0.2774,
         -0.0385,  0.3679,  0.2094, -0.5353,  1.6341,  0.1154,  1.5062,  0.1822],
        [-0.1514,  1.0416,  2.3926, -1.7163,  0.0104,  0.4489,  0.5622, -0.6685,
          2.1609, -0.3255,  1.8671,  1.0176, -0.7945,  0.4404,  0.4884, -0.7477],
        [ 0.5987,  1.7192,  0.4197, -1.3713,  0.2222, -0.3971,  1.2499,  0.0951,
          0.2134, -1.0600,  0.1008,  1.0404,  1.0911,  3.0113,  1.2188,  0.2405],
        [-0.6372, -0.3794, -0.9197, -0.1509,  1.6681,  0.3125, -0.7949, -0.4119,
          0.2921, -0.8934, -0.8438,  0.4168, -1.3112, -0.4225,  0.7332, -0.2915]])
torch.Size([3, 8, 64])
tensor([-0.0898,  1.8181, -1.5942, -2.0354, -1.5799,  0.5648,  0.3598, -0.2774,
        -0.0385,  0.3679,  0.2094, -0.5353,  1.6341,  0.1154,  1.5062,  0.1822,
        -0.1514,  1.0416,  2.3926, -1.7163,  0.0104,  0.4489,  0.5622, -0.6685,
         2.1609, -0.3255,  1.8671,  1.0176, -0.7945,  0.4404,  0.4884, -0.7477,
    

In [ ]:
# Adding another batch dimension in input
x1 = torch.randn(4, 2, 5)
print(x1.shape)
x2 = x1.unsqueeze(1)
print(x2.shape)
x3 = x2.expand(4, 3, 2, 5) # Assuming 3 heads
print(x3.shape)

print(x1[0])

print(x3[0,2])
print(x3[0,1])


torch.Size([4, 2, 5])
torch.Size([4, 1, 2, 5])
torch.Size([4, 3, 2, 5])
tensor([[ 1.8901,  0.3665, -0.9775,  1.1681, -0.1276],
        [-0.9777, -0.7182,  1.3502, -0.8733, -0.1250]])
tensor([[ 1.8901,  0.3665, -0.9775,  1.1681, -0.1276],
        [-0.9777, -0.7182,  1.3502, -0.8733, -0.1250]])
tensor([[ 1.8901,  0.3665, -0.9775,  1.1681, -0.1276],
        [-0.9777, -0.7182,  1.3502, -0.8733, -0.1250]])


EX2:

Train the GPT on your own dataset of choice! What other data could be fun to blabber on about?

(A fun advanced suggestion if you like: train a GPT to do addition of two numbers, i.e. a+b=c. You may find it helpful to predict the digits of c in reverse order, as the typical addition algorithm (that you're hoping it learns) would proceed right to left too.

You may want to modify the data loader to simply serve random problems and skip the generation of train.bin, val.bin.

You may want to mask out the loss at the input positions of a+b that just specify the problem using y=-1 in the targets (see CrossEntropyLoss ignore_index).

Does your Transformer learn to add?

Once you have this, swole doge project: build a calculator clone in GPT, for all of +-*/. Not an easy problem. You may need Chain of Thought traces.)

In [ ]:
# Generate data
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

examples = 200_000
lines = []
for _ in range(examples):
  i, j = random.randint(0, 100), random.randint(0, 100)
  lines.append(f"{i}+{j}={i+j}")

text = "\n".join(lines)

chars = list(sorted(set(text)))
vocab_size = len(chars)
# vocab_size,chars # (13, ['\n', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '='])


itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

data = torch.tensor(encode(text), dtype=torch.int64)

# Defining data split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


block_size = 8
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):
  if split == 'train':
    data = train_data
  elif split == 'val':
    data = val_data

  ix = torch.randint(0, data.shape[0]-block_size, (batch_size,))
  list_X = [data[i: i+block_size] for i in ix] # 32 list elements with each element of size 8
  X = torch.stack(list_X, dim=0) # [32, 8]
  Y = torch.stack([data[i+1: i+block_size+1] for i in ix], dim=0) # [32, 8]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train', 'val']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']} Eval loss {losses['val']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln(x)) # [B, T, C]
    x = x + self.ffwd(self.ln(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...7] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, input, num_tokens=200):
    # Initial input [1, T]
    out = []
    for _ in range(num_tokens):
      logits, _ = self(input) # [1, T, 65]
      last_T = logits[:, -1, :] # [1, 65]
      probs = F.softmax(last_T, dim=1)
      next_idx = torch.multinomial(probs, num_samples=1) # [1, 1]
      next_idx_item = next_idx.item()
      if next_idx_item == 0: # \n
        break
      out.append(next_idx_item)
      input = torch.cat((input[:,1:], next_idx), dim=1) # [1, T-1]+[1, 1] -> [1, T]

    print(f"=========== Generated output ==========\n")
    print("".join([itos[i] for i in out]))



model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)



DEVICE is cpu
Train loss 2.7266297340393066 Eval loss 2.72916316986084
Train loss 1.690637230873108 Eval loss 1.690017580986023
Train loss 1.6659146547317505 Eval loss 1.662216067314148
Train loss 1.6494559049606323 Eval loss 1.6506297588348389
Train loss 1.6399104595184326 Eval loss 1.6379196643829346
Train loss 1.6253302097320557 Eval loss 1.621762752532959
Train loss 1.6182122230529785 Eval loss 1.626038670539856
Train loss 1.6151982545852661 Eval loss 1.6148161888122559
Train loss 1.6119318008422852 Eval loss 1.6143511533737183
Train loss 1.6088600158691406 Eval loss 1.6118743419647217
Train loss 1.6040512323379517 Eval loss 1.600644588470459
Train loss 1.5992767810821533 Eval loss 1.5974721908569336
Train loss 1.6031744480133057 Eval loss 1.6054264307022095
Train loss 1.5979359149932861 Eval loss 1.6028255224227905
Train loss 1.5925242900848389 Eval loss 1.592429757118225
Train loss 1.5958151817321777 Eval loss 1.5952907800674438
Train loss 1.5898436307907104 Eval loss 1.588417410

In [ ]:
question = torch.tensor([encode(f"2+3=")])
print(question.shape)
question = F.pad(question, (block_size-question.shape[1], 0), value=0)
#question, question.shape
#start_phrase = torch.zeros((1, block_size), dtype=torch.long, device=device)
model.generate(question, num_tokens=1_000)

torch.Size([1, 4])
=========== Generated output ==========

2


In [ ]:
question = torch.tensor([encode(f"5\n")])
print(question.shape)
question = F.pad(question, (0, block_size-question.shape[1]), value=0)
question, question.shape

torch.Size([1, 2])


(tensor([[7, 0, 0, 0, 0, 0, 0, 0]]), torch.Size([1, 8]))

In [ ]:
import torch
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

In [ ]:
# Generate data
import random
import torch
import torch.nn as nn
import torch.nn.functional as F



chars = ['\n', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '.']
vocab_size = len(chars)

PAD_TOKEN = "."

itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
PAD_idx = stoi[PAD_TOKEN]

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

examples = 200_000
data = []

for _ in range(examples):
  i, j = random.randint(0, 100), random.randint(0, 100)
  data.append(encode(f"{i}+{j}={i+j}\n"))

# example = "3+5=8"
# tokens = encode(example)
# [3, 10, 5, 11, 8, 12, 12, 12, 12, 12]
#  3  +  5  =  8  .   .   .   .   .

# X = tokens[:-1]   # length 9
# Y = tokens[1:]    # length 9


block_size = 12
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

def get_batch(split='train', batch_size=32):


  ix = torch.randint(0, len(data), (batch_size,))
  # Build X
  datalists = []
  for i in ix:
    tokens = data[i]
    tokens += [PAD_idx]*(block_size-len(tokens) + 1) # (block_size + 1)
    datalists.append(torch.tensor(tokens))
  list_X = [datalist[:-1] for datalist in datalists] # [32, 12] [0-11]
  X = torch.stack(list_X, dim=0) # [32, 12] [1-12]
  list_Y = [datalist[1:] for datalist in datalists]
  Y = torch.stack(list_Y, dim=0) # [32, 12]
  return X.to(device), Y.to(device)



@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln1 = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln1(x)) # [B, T, C]
    x = x + self.ffwd(self.ln2(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...T-1] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      #logits = logits.view(B*T, C)
      #targets = targets.view(B*T)
      #loss = F.cross_entropy(logits, targets, ignore_index=PAD_idx)

      # Mask — only compute loss on tokens AFTER '='
      eq_mask = (input == stoi['=']).cumsum(dim=1)  # 0 before '=', 1+ after
      eq_mask = (eq_mask >= 1) & (input != stoi['='])  # exclude '=' itself
      # eq_mask is [B, T] boolean, True only for answer tokens

      # Flatten
      logits_flat = logits.view(B*T, C)
      targets_flat = targets.view(B*T)
      mask_flat = eq_mask.view(B*T)

      # Set non-answer targets to PAD_idx so cross_entropy ignores them
      targets_flat = targets_flat.masked_fill(~mask_flat, PAD_idx)

      loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=PAD_idx)

    return logits, loss


model = BigramModel()
model.to(device)
# Training the model
steps = 20_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)


for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()
  scheduler.step()

  if i % 500 == 0:
    evaluate_loss(eval_iters)




DEVICE is cpu
Train loss 2.6840648651123047
Train loss 1.115474820137024
Train loss 0.9727174639701843
Train loss 0.7704116702079773
Train loss 0.2501547932624817
Train loss 0.15807399153709412
Train loss 0.0958620235323906
Train loss 0.08721490949392319
Train loss 0.03950769826769829
Train loss 0.035058554261922836
Train loss 0.07223719358444214
Train loss 0.06588055938482285
Train loss 0.06790100783109665
Train loss 0.023961620405316353
Train loss 0.02168324403464794
Train loss 0.021683115512132645
Train loss 0.005433808546513319
Train loss 0.022577965632081032
Train loss 0.007053073029965162
Train loss 0.006976536009460688
Train loss 0.017956428229808807
Train loss 0.001240486977621913
Train loss 0.0008223768090829253
Train loss 0.005540960468351841
Train loss 0.002647167071700096
Train loss 0.0014304000651463866
Train loss 0.0008237536530941725
Train loss 0.000974306371062994
Train loss 0.0004186919250059873
Train loss 0.00031662583933211863
Train loss 0.00021784260752610862
Train 

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
device='cpu'
def generate(input_str):
    model.eval()
    tokens = encode(input_str)
    x_tokens = [PAD_idx] * (block_size - len(tokens)) + tokens
    x = torch.tensor(x_tokens).unsqueeze(0).to(device)  # [1, T]

    max_digits=40

    out = []
    with torch.no_grad():
      for _ in range(max_digits):
        logits, _ = model(x)                        # [1, T, vocab_size]
        next_token = logits[0,-1].argmax().item()   # [T]

        if next_token == stoi['\n']:
          break
        tokens.append(next_token)
        x_tokens = [PAD_idx] * (block_size - len(tokens)) + tokens
        x = torch.tensor(x_tokens).unsqueeze(0).to(device)  # [1, T]

    # Extract answer after '='
    answer = "".join(itos[t] for t in tokens).split("=")[-1]
    print(f"Input: {input_str}  Answer: {answer}")

# Usage
generate("3+5=")
generate("1+2=")
generate("10+1=")
generate("12+23=")
generate("99+100=")

Input: 3+5=  Answer: 
Input: 1+2=  Answer: 
Input: 10+1=  Answer: 
Input: 12+23=  Answer: 
Input: 99+100=  Answer: 


In [ ]:
def evaluate_accuracy(n=200):
    correct = 0
    for _ in range(n):
        i, j = random.randint(0, 100), random.randint(0, 100)
        input_str = f"{i}+{j}="
        tokens = encode(input_str)
        padded = [PAD_idx] * (block_size - len(tokens)) + tokens
        x = torch.tensor(padded, dtype=torch.long).unsqueeze(0).to(device)

        generated = []
        model.eval()
        with torch.no_grad():
            for _ in range(4):
                logits, _ = model(x)
                next_tok = logits[0, -1].argmax().item()
                if next_tok == stoi['\n']:
                    break
                generated.append(itos[next_tok])
                tokens.append(next_tok)
                padded = [PAD_idx] * (block_size - len(tokens)) + tokens
                x = torch.tensor(padded, dtype=torch.long).unsqueeze(0).to(device)

        pred = "".join(generated).strip()
        correct += (pred == str(i + j))

    print(f"Accuracy: {correct}/{n} = {correct/n*100:.1f}%")

evaluate_accuracy()

Accuracy: 0/200 = 0.0%


### Trying separate X and Y
input:  "12+7="
target: "19"

In [ ]:
# Generate data
import random
import torch
import torch.nn as nn
import torch.nn.functional as F


chars = ['\n', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '.']
vocab_size = len(chars)

PAD_TOKEN = "."

itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
PAD_idx = stoi[PAD_TOKEN]

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

block_size = 12
n_embd = 32

examples = 500_000
data_X = []
data_Y = []

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE is {device}")

for _ in range(examples):
  i, j = random.randint(0, 100), random.randint(0, 100)
  x = encode(f"{i}+{j}=")
  x += [PAD_idx]*(block_size-len(x))
  y = encode(f"{i+j}\n")
  y += [PAD_idx]*(block_size-len(y))
  data_X.append(torch.tensor(x))
  data_Y.append(torch.tensor(y))

def get_batch(split='train', batch_size=32):
  ix = torch.randint(0, len(data_X), (batch_size,))
  list_X = [data_X[i] for i in ix] # [32, 12] [0-11]
  X = torch.stack(list_X, dim=0) # [32, 12] [1-12]
  list_Y = [data_Y[i] for i in ix]
  Y = torch.stack(list_Y, dim=0) # [32, 12]
  return X.to(device), Y.to(device)



@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    #wei = wei.masked_fill(self.tril==0, -torch.inf)
    wei = wei.masked_fill(self.tril[:T,:T] == 0, -torch.inf) # Need to use this because can't pass variable length input during inference otherwise
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln1 = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln1(x)) # [B, T, C]
    x = x + self.ffwd(self.ln2(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...T-1] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets, ignore_index=PAD_idx)

    return logits, loss


model = BigramModel()
model.to(device)
# Training the model
steps = 50_000
lr = 3e-4
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)


for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()
  scheduler.step()

  if i % 1000 == 0:
    evaluate_loss(eval_iters)




DEVICE is cuda
Train loss 2.8156261444091797
Train loss 1.567152976989746
Train loss 1.5640590190887451
Train loss 1.544191837310791
Train loss 1.547873854637146
Train loss 1.5480754375457764
Train loss 1.54056978225708
Train loss 1.5451247692108154
Train loss 1.5484808683395386
Train loss 1.5489108562469482
Train loss 1.547783613204956
Train loss 1.547593116760254
Train loss 1.5483250617980957
Train loss 1.5420899391174316
Train loss 1.5423566102981567
Train loss 1.5424271821975708
Train loss 1.5428357124328613
Train loss 1.5486268997192383
Train loss 1.541407823562622
Train loss 1.541349172592163
Train loss 1.545931100845337
Train loss 1.5415234565734863
Train loss 1.5451593399047852
Train loss 1.5446909666061401
Train loss 1.534967303276062
Train loss 1.544521450996399
Train loss 1.5451937913894653
Train loss 1.5405174493789673
Train loss 1.5443800687789917
Train loss 1.5385669469833374
Train loss 1.5404765605926514
Train loss 1.5396357774734497
Train loss 1.5452287197113037
Train l

In [ ]:
def generate(input_str, max_new_tokens=20):
    model.eval()
    device = next(model.parameters()).device

    # Start with ONLY the input (e.g. "12+7=")
    tokens = encode(input_str)
    x = torch.tensor(tokens, device=device).unsqueeze(0)  # [1, T]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Crop to block size (important for long sequences)
            x_cond = x[:, -block_size:]

            logits, _ = model(x_cond)

            # Take last position prediction
            logits_last = logits[0, -1]

            # Greedy decode (best for math)
            next_token = torch.argmax(logits_last).item()
            print(f"Next token {next_token}")

            # Stop at newline
            if next_token == stoi['\n']:
                break

            # Append token
            tokens.append(next_token)

            # Update input
            x = torch.tensor(tokens, device=device).unsqueeze(0)

    decoded = "".join(itos[t] for t in tokens)

    # Extract answer
    if "=" in decoded:
        answer = decoded.split("=")[-1]
    else:
        answer = decoded

    print(f"Input: {input_str}  Answer: {answer}")

# Usage
generate("3+5=")
generate("1+2=")
generate("10+1=")
generate("12+23=")
generate("99+100=")

Next token 0
Input: 3+5=  Answer: 
Next token 0
Input: 1+2=  Answer: 
Next token 0
Input: 10+1=  Answer: 
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Input: 12+23=  Answer: 11111111111111111111
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Next token 3
Input: 99+100=  Answer: 11111111111111111111


### Trying with reversing the answer [WORKING MODEL]

In [ ]:
# Generate data and define model
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

chars = ['\n', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '.']
vocab_size = len(chars)

PAD_TOKEN = "."

itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
PAD_idx = stoi[PAD_TOKEN]

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

examples = 500_000
data = []

for _ in range(examples):
  i, j = random.randint(0, 100), random.randint(0, 100)
  data.append(encode(f"{i}+{j}={str(i+j)[::-1]}\n")) # Adding reverse and \n as end token helped but regular + '\n' was not working at all

block_size = 12
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split='train', batch_size=32):
  ix = torch.randint(0, len(data), (batch_size,))
  # Build X
  datalists = []
  for i in ix:
    tokens = data[i]
    tokens += [PAD_idx]*(block_size-len(tokens) + 1) # (block_size + 1)
    datalists.append(torch.tensor(tokens))
  list_X = [datalist[:-1] for datalist in datalists] # [32, 12] [0-11]
  X = torch.stack(list_X, dim=0) # [32, 12] [1-12]
  list_Y = [datalist[1:] for datalist in datalists]
  Y = torch.stack(list_Y, dim=0) # [32, 12]
  return X.to(device), Y.to(device)



@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril[:T,:T]==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln1 = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln1(x)) # [B, T, C]
    x = x + self.ffwd(self.ln2(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...T-1] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      #logits = logits.view(B*T, C)
      #targets = targets.view(B*T)
      #loss = F.cross_entropy(logits, targets, ignore_index=PAD_idx)

      # We want loss only on tokens AFTER '=' in X

      eq_mask = (input == stoi['='])          # [B, T]
      after_eq = eq_mask.cumsum(dim=1)        # 0 before '=', >0 after
      mask = (after_eq > 0) & (targets != PAD_idx)

      # Ignore everything outside the mask
      masked_targets = targets.masked_fill(~mask, PAD_idx)

      loss = F.cross_entropy(
          logits.view(B*T, C),
          masked_targets.view(B*T),
          ignore_index=PAD_idx
      )

    return logits, loss


model = BigramModel()
model.to(device)

print(device)

cpu


In [ ]:
# Training the model
steps = 15_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()
  scheduler.step()

  if i % 1_000 == 0:
    evaluate_loss(eval_iters)

Train loss 2.699575424194336
Train loss 0.7255740165710449
Train loss 0.07829679548740387
Train loss 0.04043871536850929
Train loss 0.014317413792014122
Train loss 0.01556949969381094
Train loss 0.05446089804172516
Train loss 0.000449950632173568
Train loss 0.000309229624690488
Train loss 0.00021147070219740272
Train loss 0.00014838521019555628
Train loss 0.00011604947212617844
Train loss 9.080918243853375e-05
Train loss 7.527261914219707e-05
Train loss 7.045584061415866e-05


In [ ]:
def generate(input_str):

    # Start with ONLY the input (e.g. "12+7=")
    tokens = encode(input_str)

    # ** NO NEED FOR PADDING **
    x = torch.tensor(tokens, device=device).unsqueeze(0)  # [1, T]

    max_new_tokens = 4
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Crop to block size (important for long sequences)
            x_cond = x[:, -block_size:]

            logits, _ = model(x_cond)

            # Take last position prediction
            logits_last = logits[0, -1]

            # Greedy decode (best for math)
            next_token = torch.argmax(logits_last).item()
            # Stop at newline
            if next_token in [stoi['\n'], stoi['+'], stoi['='], stoi['.']]:
                break

            # Append token
            tokens.append(next_token)

            # Update input
            x = torch.tensor(tokens, device=device).unsqueeze(0)

    decoded = "".join(itos[t] for t in tokens)

    # Extract answer
    if "=" in decoded:
        answer = decoded.split("=")[-1][::-1] # reversing
    else:
        answer = decoded

    print(f"Input: {input_str}  Decoded: {decoded} Answer: {answer}")
    return answer

"""
# Need to limit digits otherwise
Input: 3+5=  Decoded: 3+5=81675191111111111111 Answer: 11111111111119157618
Input: 1+2=  Decoded: 1+2=31203121111111111111 Answer: 11111111111112130213
Input: 10+1=  Decoded: 10+1=11111211111111111111 Answer: 11111111111111211111
Input: 12+23=  Decoded: 12+23=53111111111111111111 Answer: 11111111111111111135
Input: 99+100=  Decoded: 99+100=99191111111111111111 Answer: 11111111111111119199
Input: 9+9=  Decoded: 9+9=81019191111111111111 Answer: 11111111111119191018
Input: 100+100=  Decoded: 100+100=00211111111111111111 Answer: 11111111111111111200
"""

# Usage
generate("3+5=")
generate("1+2=")
generate("10+1=")
generate("12+23=")
generate("99+100=")
generate("9+9=")
generate("100+100=")
generate("67+93=")
generate("99+87=")
generate("87+12=")
generate("101+13=") # Example of model merely memorising
generate("123+540=")

Input: 3+5=  Decoded: 3+5=8 Answer: 8
Input: 1+2=  Decoded: 1+2=3 Answer: 3
Input: 10+1=  Decoded: 10+1=11 Answer: 11
Input: 12+23=  Decoded: 12+23=53 Answer: 35
Input: 99+100=  Decoded: 99+100=991 Answer: 199
Input: 9+9=  Decoded: 9+9=81 Answer: 18
Input: 100+100=  Decoded: 100+100=002 Answer: 200
Input: 67+93=  Decoded: 67+93=061 Answer: 160
Input: 99+87=  Decoded: 99+87=681 Answer: 186
Input: 87+12=  Decoded: 87+12=99 Answer: 99
Input: 101+13=  Decoded: 101+13=311 Answer: 113
Input: 123+540=  Decoded: 123+540=51 Answer: 15


'15'

In [ ]:
def evaluate_accuracy(n=200):
    correct = 0
    for _ in range(n):
        i, j = random.randint(101, 999), random.randint(101, 999)
        input_str = f"{i}+{j}="
        pred = generate(input_str)
        correct += (pred == str(i + j))

    print(f"Accuracy: {correct}/{n} = {correct/n*100:.1f}%")

evaluate_accuracy()

Input: 448+707=  Decoded: 448+707=71 Answer: 17
Input: 400+784=  Decoded: 400+784=71 Answer: 17
Input: 275+396=  Decoded: 275+396=71 Answer: 17
Input: 718+306=  Decoded: 718+306=51 Answer: 15
Input: 946+901=  Decoded: 946+901=5 Answer: 5
Input: 612+242=  Decoded: 612+242=41 Answer: 14
Input: 315+301=  Decoded: 315+301=31 Answer: 13
Input: 107+725=  Decoded: 107+725=71 Answer: 17
Input: 456+502=  Decoded: 456+502=4 Answer: 4
Input: 421+717=  Decoded: 421+717=81 Answer: 18
Input: 224+594=  Decoded: 224+594=51 Answer: 15
Input: 264+224=  Decoded: 264+224=41 Answer: 14
Input: 728+627=  Decoded: 728+627=71 Answer: 17
Input: 702+505=  Decoded: 702+505=41 Answer: 14
Input: 520+917=  Decoded: 520+917=81 Answer: 18
Input: 678+797=  Decoded: 678+797=51 Answer: 15
Input: 139+593=  Decoded: 139+593=41 Answer: 14
Input: 652+504=  Decoded: 652+504=4 Answer: 4
Input: 755+944=  Decoded: 755+944=51 Answer: 15
Input: 878+426=  Decoded: 878+426=71 Answer: 17
Input: 701+909=  Decoded: 701+909=81 Answer: 1

### Notes from first working model
- The model actually started working with reversing output
- However, it was failing to know when to stop without adding an end token.
e.g.

_Input: 12+23=  Decoded: 12+23=53111111111111111111 Answer: 11111111111111111135_

I chose '\n'. Also note that adding '\n' in regular order and not reverse order worked

- Chosing how to build X and Y matters
  
  What worked X [1,2,+,2,3,=,3,5]
              Y [2,+,2,3,=,3,5,.]

  
  What didn't work X [1, 2, +, 2, 3, =, ., .]
                   Y [3, 5, ., ., ., ., ., .]

 - Model most likely memorizes the answers instead of generalizing evident by having out of range numbers in inference _e.g. Input: 101+13=  Decoded: 101+13=311 Answer: 113_

 Evaluate accuracy results on 200 out of range numbers: **0 %**

### [PROBLEM] Trying to predict well on out of range inputs

In [ ]:
# SOLUTION 1: Training on bigger range

# Generate data and define model
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

chars = ['\n', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '.']
vocab_size = len(chars)

PAD_TOKEN = "."

itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
PAD_idx = stoi[PAD_TOKEN]

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

examples = 500_000
data = []

for _ in range(examples):
  i, j = random.randint(0, 10_000), random.randint(0, 10_000)
  data.append(encode(f"{i}+{j}={str(i+j)[::-1]}\n")) # Adding reverse and \n as end token helped but regular + '\n' was not working at all

block_size = 22 # HAD TO BUMP THIS
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split='train', batch_size=32):
  ix = torch.randint(0, len(data), (batch_size,))
  # Build X
  datalists = []
  for i in ix:
    tokens = data[i]
    tokens += [PAD_idx]*(block_size-len(tokens) + 1) # (block_size + 1)
    datalists.append(torch.tensor(tokens))
  list_X = [datalist[:-1] for datalist in datalists] # [32, 12] [0-11]
  X = torch.stack(list_X, dim=0) # [32, 12] [1-12]
  list_Y = [datalist[1:] for datalist in datalists]
  Y = torch.stack(list_Y, dim=0) # [32, 12]
  return X.to(device), Y.to(device)



@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril[:T,:T]==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln1 = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln1(x)) # [B, T, C]
    x = x + self.ffwd(self.ln2(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...T-1] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      #logits = logits.view(B*T, C)
      #targets = targets.view(B*T)
      #loss = F.cross_entropy(logits, targets, ignore_index=PAD_idx)

      # We want loss only on tokens AFTER '=' in X

      eq_mask = (input == stoi['='])          # [B, T]
      after_eq = eq_mask.cumsum(dim=1)        # 0 before '=', >0 after
      mask = (after_eq > 0) & (targets != PAD_idx)

      # Ignore everything outside the mask
      masked_targets = targets.masked_fill(~mask, PAD_idx)

      loss = F.cross_entropy(
          logits.view(B*T, C),
          masked_targets.view(B*T),
          ignore_index=PAD_idx
      )

    return logits, loss


model = BigramModel()
model.to(device)

print(device)

cpu


In [ ]:
# Training the model
steps = 30_000
lr = 3e-4
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()
  scheduler.step()

  if i % 1_000 == 0:
    evaluate_loss(eval_iters)

Train loss 2.768139600753784
Train loss 1.686152696609497
Train loss 1.6754745244979858
Train loss 1.6725108623504639
Train loss 1.4946423768997192
Train loss 1.3587613105773926
Train loss 1.2766809463500977
Train loss 1.2526966333389282
Train loss 1.2471730709075928
Train loss 1.2344950437545776
Train loss 1.2401126623153687
Train loss 1.235724687576294
Train loss 1.2302619218826294
Train loss 1.229873538017273
Train loss 1.1280641555786133
Train loss 0.9491937160491943
Train loss 0.9217921495437622
Train loss 0.9081655740737915
Train loss 0.9049195647239685
Train loss 0.8989173769950867
Train loss 0.8928640484809875
Train loss 0.8917045593261719
Train loss 0.8922269940376282
Train loss 0.8918941020965576
Train loss 0.8928581476211548
Train loss 0.888997495174408
Train loss 0.8888314962387085
Train loss 0.892974317073822
Train loss 0.8861604928970337
Train loss 0.891506552696228


In [ ]:
def generate(input_str):

    # Start with ONLY the input (e.g. "12+7=")
    tokens = encode(input_str)

    # ** NO NEED FOR PADDING **
    x = torch.tensor(tokens, device=device).unsqueeze(0)  # [1, T]

    max_new_tokens = 4
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Crop to block size (important for long sequences)
            x_cond = x[:, -block_size:]

            logits, _ = model(x_cond)

            # Take last position prediction
            logits_last = logits[0, -1]

            # Greedy decode (best for math)
            next_token = torch.argmax(logits_last).item()
            # Stop at newline
            if next_token in [stoi['\n'], stoi['+'], stoi['='], stoi['.']]:
                break

            # Append token
            tokens.append(next_token)

            # Update input
            x = torch.tensor(tokens, device=device).unsqueeze(0)

    decoded = "".join(itos[t] for t in tokens)

    # Extract answer
    if "=" in decoded:
        answer = decoded.split("=")[-1][::-1] # reversing
    else:
        answer = decoded

    print(f"Input: {input_str}  Decoded: {decoded} Answer: {answer}")
    return answer

# Usage
generate("3+5=")
generate("1+2=")
generate("10+1=")
generate("12+23=")
generate("99+100=")
generate("9+9=")
generate("100+100=")
generate("67+93=")
generate("99+87=")
generate("87+12=")
generate("101+13=")
generate("123+540=")
generate("1123+1540=")
generate("10001+15000=")

Input: 3+5=  Decoded: 3+5=44 Answer: 44
Input: 1+2=  Decoded: 1+2=74 Answer: 47
Input: 10+1=  Decoded: 10+1=5 Answer: 5
Input: 12+23=  Decoded: 12+23=4717 Answer: 7174
Input: 99+100=  Decoded: 99+100=659 Answer: 956
Input: 9+9=  Decoded: 9+9=7434 Answer: 4347
Input: 100+100=  Decoded: 100+100=211 Answer: 112
Input: 67+93=  Decoded: 67+93=4767 Answer: 7674
Input: 99+87=  Decoded: 99+87=4488 Answer: 8844
Input: 87+12=  Decoded: 87+12=4478 Answer: 8744
Input: 101+13=  Decoded: 101+13=215 Answer: 512
Input: 123+540=  Decoded: 123+540=354 Answer: 453
Input: 1123+1540=  Decoded: 1123+1540=0762 Answer: 2670
Input: 10001+15000=  Decoded: 10001+15000=2 Answer: 2


'2'

In [ ]:
# SOLUTION 2: CURRICULUM LEARNING

# Generate data and define model
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

chars = ['\n', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '.']
vocab_size = len(chars)
itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
PAD_TOKEN = "."
PAD_idx = stoi[PAD_TOKEN]

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

block_size = 22 # 18+4(buffer)
n_embd = 32

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_max_digits(step, total_steps):
    # Linearly increase difficulty over training
    progress = step / total_steps
    if progress < 0.25:
        return 10      # 1 digit numbers
    elif progress < 0.5:
        return 100     # 2 digit numbers
    elif progress < 0.75:
        return 1000    # 3 digit numbers
    else:
        return 10000   # 4 digit numbers

def get_batch(step, total_steps, batch_size=32):
    max_val = get_max_digits(step, total_steps)
    list_X, list_Y = [], []
    for _ in range(batch_size):
        i = random.randint(0, max_val)
        j = random.randint(0, max_val)
        tokens = encode(f"{i}+{j}={str(i+j)[::-1]}\n")
        tokens += [PAD_idx] * (block_size - len(tokens))
        list_X.append(torch.tensor(tokens[:-1]))
        list_Y.append(torch.tensor(tokens[1:]))
    X = torch.stack(list_X, dim=0) # [32, 22]
    Y = torch.stack(list_Y, dim=0) # [32, 22]
    return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters, step, total_steps):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(step, total_steps)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)
    # Intuition for projection:
    # The projection layer is needed not just to match dimensions,
    # but to mix information across heads and align the output with the input space
    # so the residual connection works properly.
    # Think of it like:

    # Heads = specialists
    # Projection = team meeting where results are combined
    # Without projection:
    # everyone talks, nobody integrates

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril[:T,:T]==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln1 = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln1(x)) # [B, T, C]
    x = x + self.ffwd(self.ln2(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...T-1] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      #logits = logits.view(B*T, C)
      #targets = targets.view(B*T)
      #loss = F.cross_entropy(logits, targets, ignore_index=PAD_idx)

      # We want loss only on tokens AFTER '=' in X

      eq_mask = (input == stoi['='])          # [B, T]
      after_eq = eq_mask.cumsum(dim=1)        # 0 before '=', >0 after
      mask = (after_eq > 0) & (targets != PAD_idx)

      # Ignore everything outside the mask
      masked_targets = targets.masked_fill(~mask, PAD_idx)

      loss = F.cross_entropy(
          logits.view(B*T, C),
          masked_targets.view(B*T),
          ignore_index=PAD_idx
      )

    return logits, loss


model = BigramModel()
model.to(device)

print(device)

cpu


In [ ]:
# Training the model
steps = 30_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)

for i in range(steps):
  Xb, Yb = get_batch(step=i, total_steps=steps)
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()
  scheduler.step()

  if i % 1_000 == 0:
    #evaluate_loss(eval_iters, i, steps)
    print(loss.item())

3.5561158657073975
0.00017671872046776116
7.569172157673165e-05
4.96997672598809e-05
2.7427688110037707e-05
2.1838155589648522e-05
1.2188977962068748e-05
6.91260356688872e-06
0.0011342627694830298
0.00042933382792398334
0.00012567764497362077
0.00010266087338095531
0.00016947132826317102
0.00011265603097854182
7.075213943608105e-05
7.199389457702637
0.03817934915423393
0.0014234556583687663
0.04146081954240799
0.0008175881812348962
0.052550606429576874
0.0035838796757161617
0.00021645330707542598
1.2266266345977783
0.9671826958656311
0.8862070441246033
0.8581359386444092
0.8621767163276672
0.8322659134864807
0.8479833602905273


In [ ]:
generate("3+5=")
generate("1+2=")
generate("10+1=")
generate("12+23=")
generate("99+100=")
generate("9+9=")
generate("100+100=")
generate("67+93=")
generate("99+87=")
generate("87+12=")
generate("101+13=")
generate("123+540=")
generate("1123+1540=")
generate("10001+15000=")

Input: 3+5=  Decoded: 3+5=6 Answer: 6
Input: 1+2=  Decoded: 1+2=4 Answer: 4
Input: 10+1=  Decoded: 10+1=01 Answer: 10
Input: 12+23=  Decoded: 12+23=521 Answer: 125
Input: 99+100=  Decoded: 99+100=9912 Answer: 2199
Input: 9+9=  Decoded: 9+9=8116 Answer: 6118
Input: 100+100=  Decoded: 100+100=012 Answer: 210
Input: 67+93=  Decoded: 67+93=601 Answer: 106
Input: 99+87=  Decoded: 99+87=6611 Answer: 1166
Input: 87+12=  Decoded: 87+12=001 Answer: 100
Input: 101+13=  Decoded: 101+13=321 Answer: 123
Input: 123+540=  Decoded: 123+540=366 Answer: 663
Input: 1123+1540=  Decoded: 1123+1540=3766 Answer: 6673
Input: 10001+15000=  Decoded: 10001+15000=5 Answer: 5


'5'

### SOLUTION 3: CHAIN OF THOUGHT

In [ ]:
# Generate data and define model
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

chars = ['\n', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '.', '[', ']', ';', 'c']
vocab_size = len(chars)

PAD_TOKEN = "."

itos = {i:c for i,c in enumerate(chars)}
stoi = {c:i for i,c in enumerate(chars)}
PAD_idx = stoi[PAD_TOKEN]

encode = lambda s: [stoi[c] for c in s]
decode = lambda x: [itos[i] for i in x]

examples = 500_000
data = []

max_num = 1_000
max_digits = 4 # (999+999=1998)

def make_cot(i, j):
    si = str(i).zfill(max_digits)  # pad to max digits i.e. 100, 99 -> 099
    sj = str(j).zfill(max_digits)

    steps = []
    carry = 0
    for a, b in zip(reversed(si), reversed(sj)):
        in_carry = carry          # save incoming carry FIRST
        total = int(a) + int(b) + carry
        digit = total % 10
        carry = total // 10
        steps.append(f"{a}+{b}+{in_carry}={digit}c{carry}")

    answer = str(i + j)[::-1]  # still reverse final answer
    scratchpad = ";".join(steps)
    return f"{i}+{j}=[{scratchpad}]={answer}\n"

for _ in range(examples):
  i, j = random.randint(0, max_num), random.randint(0, max_num)
  data.append(encode(make_cot(i,j))) # Adding reverse and \n as end token helped but regular + '\n' was not working at all

block_size = 64
n_embd = 128 # Bumping this up

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def get_batch(split='train', batch_size=32):
  ix = torch.randint(0, len(data), (batch_size,))
  # Build X
  datalists = []
  for i in ix:
    tokens = data[i]
    tokens += [PAD_idx]*(block_size-len(tokens) + 1) # (block_size + 1)
    datalists.append(torch.tensor(tokens))
  list_X = [datalist[:-1] for datalist in datalists] # [32, 12] [0-11]
  X = torch.stack(list_X, dim=0) # [32, 12] [1-12]
  list_Y = [datalist[1:] for datalist in datalists]
  Y = torch.stack(list_Y, dim=0) # [32, 12]
  return X.to(device), Y.to(device)

@torch.no_grad()
def evaluate_loss(eval_iters):
  losses = {}
  model.eval() # Turns OFF model training behaviour e.g. Dropout layer or BatchNorm layer
  for split in ['train']:
    loss = torch.zeros(eval_iters)
    for i in range(eval_iters):
      Xb, Yb = get_batch(split)
      logits, lossi = model(Xb, Yb)
      loss[i] = lossi
    losses[split] = loss.mean()
  model.train() # Turns ON model training behaviour
  print(f"Train loss {losses['train']}")

class MultiHeadAttention(nn.Module):
  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([AttnHead(head_size) for _ in range(num_heads)])
    self.projection = nn.Linear(n_embd, n_embd)

  def forward(self, x):
    out = [head(x) for head in self.heads]
    out = torch.cat(out, dim=-1)
    out = self.projection(out)
    return out

class AttnHead(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.Q = nn.Linear(n_embd, head_size, bias=False) # Stores [out_features, in_features] and X @ W.T
    self.K = nn.Linear(n_embd, head_size, bias=False)
    self.V = nn.Linear(n_embd, head_size, bias=False)
    #https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, input):
    # input [B, T, n_embd]
    B,T,C = input.shape
    query = self.Q(input) # [n_embd, n_embd] broadcasted to [B, n_embd, n_embd]; [B, T, n_embd] @ [B, n_embd, n_embd]
    key = self.K(input) # [B, T, n_embd]
    value = self.V(input) # [B, T, n_embd]
    # print(f"I: {input.shape} K: {key.shape} Q: {query.shape} KT: {torch.transpose(key, -2, -1).shape}")

    wei = query @ torch.transpose(key, -2, -1) # [B, T, n_embd] @ [B, n_embd, T] -> [B, T, T]
    wei = wei * (C**-0.5) # Scaling down
    # Until this wei is initial affinities based on data

    wei = wei.masked_fill(self.tril[:T,:T]==0, -torch.inf)
    wei = F.softmax(wei, dim=-1)
    out = wei @ value # [B,T,T] @ [B,T,n_embd] -> [B,T,n_embd]
    return out

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd) # Additional projection
    )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.sa_heads = MultiHeadAttention(4, n_embd//4)
    self.ffwd = FeedForward()
    self.ln1 = nn.LayerNorm(n_embd) # [Mean and variance taken over 32 numbers i.e. n_embd]
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # [Add & Norm] as in Transformers diagram
    x = x + self.sa_heads(self.ln1(x)) # [B, T, C]
    x = x + self.ffwd(self.ln2(x)) # [B,T,C]
    return x


class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding_table = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embd)
    self.position_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd),
        Block(n_embd),
        Block(n_embd),
        nn.LayerNorm(n_embd)
    )
    self.lm_head = nn.Linear(in_features=n_embd, out_features=vocab_size)

  def forward(self, input, targets = None):
    # input [B,T] (batch, time)
    # targets: [B, T]
    B, T = input.shape
    token_embd = self.embedding_table(input) # (B, T, embd) -> (batch, time, channels) -> channels is embedding size here
    pos_input = torch.arange(T, device=device).repeat(B,1) # [B, T] [[0,1,2...T-1] ... 8 batch]
    pos_embd = self.position_table(pos_input) # (B, T, embd)
    # Alternatively can do
    # pos_embd = self.position_table(torch.arange(T)) # [T, embd] (would work in next step because of broadcasting)
    x = token_embd + pos_embd
    x = self.blocks(x)
    logits = self.lm_head(x) # (B, T, vocab_size)
    loss = None
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      #logits = logits.view(B*T, C)
      #targets = targets.view(B*T)
      #loss = F.cross_entropy(logits, targets, ignore_index=PAD_idx)

      # We want loss only on tokens AFTER '=' in X

      eq_mask = (input == stoi['='])          # [B, T]
      after_eq = eq_mask.cumsum(dim=1)        # 0 before '=', >0 after
      mask = (after_eq > 0) & (targets != PAD_idx)

      # Ignore everything outside the mask
      masked_targets = targets.masked_fill(~mask, PAD_idx)

      loss = F.cross_entropy(
          logits.view(B*T, C),
          masked_targets.view(B*T),
          ignore_index=PAD_idx
      )

    return logits, loss


model = BigramModel()
model.to(device)

print(device)

cpu


In [ ]:
# Training the model
steps = 15_000
lr = 1e-3
eval_iters = 200
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps)

for i in range(steps):
  Xb, Yb = get_batch()
  logits, loss = model(Xb, Yb)

  # Back pass
  optimizer.zero_grad()
  loss.backward()

  # Update
  optimizer.step()
  scheduler.step()

  if i % 1_000 == 0:
    evaluate_loss(eval_iters)

Train loss 2.580414056777954
Train loss 0.017485391348600388
Train loss 0.003455429570749402
Train loss 0.0015251918230205774
Train loss 0.0006476010312326252
Train loss 0.000945903651881963
Train loss 0.00022302452998701483
Train loss 2.5243538402719423e-05
Train loss 0.00020090969337616116
Train loss 0.00012487679487094283
Train loss 3.529909372446127e-05
Train loss 5.710403002012754e-06
Train loss 2.987264224429964e-06
Train loss 5.534004685614491e-06
Train loss 1.145905025623506e-05


In [ ]:
def generate(input_str, print_it=True):

    # Start with ONLY the input (e.g. "12+7=")
    tokens = encode(input_str)

    # ** NO NEED FOR PADDING **
    x = torch.tensor(tokens, device=device).unsqueeze(0)  # [1, T]

    max_new_tokens = block_size
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Crop to block size (important for long sequences)
            x_cond = x[:, -block_size:]

            logits, _ = model(x_cond)

            # Take last position prediction
            logits_last = logits[0, -1]

            # Greedy decode (best for math)
            next_token = torch.argmax(logits_last).item()
            # Stop at newline or pad token
            if next_token in [stoi['\n'], stoi['.']]:
                break

            # Append token
            tokens.append(next_token)

            # Update input
            x = torch.tensor(tokens, device=device).unsqueeze(0)

    decoded = "".join(itos[t] for t in tokens)

    # Extract answer
    if "]=" in decoded:
        answer = decoded.split("]=")[-1][::-1] # reversing
    else:
        answer = decoded
    if print_it:
      print(f"Input: {input_str}  Decoded: {decoded} Answer: {answer}")
    return answer

# Usage
generate("3+5=")
generate("1+2=")
generate("10+1=")
generate("12+23=")
generate("99+100=")
generate("9+9=")
generate("100+100=")
generate("67+93=")
generate("99+87=")
generate("87+12=")
generate("101+13=") # Example of model merely memorising
generate("123+540=")
generate("1001+49=")
#evaluate_accuracy()

Input: 3+5=  Decoded: 3+5=[3+5+0=8c0;0+0+0=0c0;0+0+0=0c0;0+0+0=0c0]=8 Answer: 8
Input: 1+2=  Decoded: 1+2=[1+2+0=3c0;0+0+0=0c0;0+0+0=0c0;0+0+0=0c0]=9 Answer: 9
Input: 10+1=  Decoded: 10+1=[0+1+0=1c0;1+0+0=1c0;0+0+0=0c0;0+0+0=0c0]=11 Answer: 11
Input: 12+23=  Decoded: 12+23=[2+3+0=5c0;1+2+0=3c0;0+0+0=0c0;0+0+0=0c0]=53 Answer: 35
Input: 99+100=  Decoded: 99+100=[9+0+0=9c0;9+0+0=9c0;0+1+0=1c0;0+0+0=0c0]=991 Answer: 199
Input: 9+9=  Decoded: 9+9=[9+9+0=8c1;0+1+1=0c1;0+1+0=0=1c0;0+0=0=0]=91 Answer: 19
Input: 100+100=  Decoded: 100+100=[0+0+0=0c0;0+0+0=0c0;1+1+0=2c0;0+0+0=0c0]=002 Answer: 200
Input: 67+93=  Decoded: 67+93=[7+3+0=0c1;6+9+1=6c1;0+0+1=1c0;0+0+0=0c0]=061 Answer: 160
Input: 99+87=  Decoded: 99+87=[9+7+0=6c1;9+8+1=8c1;0+0+1=1c0;0+0+0=0c0]=681 Answer: 186
Input: 87+12=  Decoded: 87+12=[7+2+0=9c0;8+1+0=9c0;0+0+0=0c0;0+0+0=0c0]=99 Answer: 99
Input: 101+13=  Decoded: 101+13=[1+3+0=4c0;0+1+0=1c0;1+0+0=1c0;0+0+0=0c0]=411 Answer: 114
Input: 123+540=  Decoded: 123+540=[3+0+0=3c0;2+4+0=6c0

'259'

### Tomorrow

- Curriculum learning
- Chain of thought

In [ ]:
make_cot(121,198)
make_cot(199, 233)

'199+233=[9+3+0=2c1;9+3+1=3c1;1+2+1=4c0;0+0+0=0c0]=234\n'

#### PERSONAL SCRATCHPAD FOR UNDERSTANDING

In [ ]:
# [TOY EXAMPLE] : How mask works

input_str1 = f"12+25=37"
input1 = torch.tensor(encode(input_str1))
input_str2 = f"19+21=40"
input2 = torch.tensor(encode(input_str2))
input3 = torch.stack([input1, input2], dim=0)
print("Input", input3, input3.shape)

eq_pos = (input3 == stoi['=']).int().argmax(dim=1)  # [B] index of '=' in each row
print(eq_pos, eq_pos.shape, eq_pos.unsqueeze(1), eq_pos.unsqueeze(1).shape)
# Build mask: True for positions after '='
pos = torch.arange(input3.shape[1]).unsqueeze(0)  # [1, T]
print(pos, pos.shape)
mask = pos > eq_pos.unsqueeze(1) # [1, T] > [B, 1] (Each broadcasts to [B, T] for element wise comparison)
print(mask)

Input tensor([[ 3,  4,  1,  4,  7, 12,  5,  9],
        [ 3, 11,  1,  4,  3, 12,  6,  2]]) torch.Size([2, 8])
tensor([5, 5]) torch.Size([2]) tensor([[5],
        [5]]) torch.Size([2, 1])
tensor([[0, 1, 2, 3, 4, 5, 6, 7]]) torch.Size([1, 8])
tensor([[False, False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False,  True,  True]])
